# SAM2 Motion Mask Generation

Interactive notebook for generating motion masks using SAM2 video predictor.

**Output:**
- `<data_path>/motion_masks/` — foreground=1, background=0 (grayscale PNG)
- `<data_path>/masks/` — foreground=0, background=1 (for COLMAP masking)

**Usage:**
1. Set `DATA_PATH` and SAM2 paths in Cell 2
2. Run cells sequentially
3. **Left click** on the first frame to add positive points (foreground)
4. **Right click** to add negative points (background)
5. Run propagation cell to process all frames

In [ ]:
import matplotlib
matplotlib.use('module://ipympl.backend_nbagg')

import os
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display
from PIL import Image
from glob import glob

print('Libraries loaded.')

In [ ]:
# ============================================================
# CONFIGURATION — edit DATA_PATH before running
# ============================================================

DATA_PATH = "/home/shindy/projects/hdr-4d/hdr-nsff/data/hdr-gopro/bear_thread_test"

# SAM2 repo is cloned as sam2_repo/ (not sam2/) to avoid Python package name conflict.
# Checkpoint is downloaded by setup_hdr_nsff.sh.
import pathlib
_NSFF_SCRIPTS = pathlib.Path(__file__).parent if '__file__' in dir() else pathlib.Path.cwd()
SAM2_REPO_PATH  = str(_NSFF_SCRIPTS / "sam2_repo")
SAM2_CHECKPOINT = str(_NSFF_SCRIPTS / "sam2_repo" / "checkpoints" / "sam2.1_hiera_large.pt")
SAM2_MODEL_CFG  = "configs/sam2.1/sam2.1_hiera_l.yaml"

OBJ_ID = 1

# ============================================================
import os
IMAGES_DIR    = os.path.join(DATA_PATH, 'images')
MASK_OUT_DIR  = os.path.join(DATA_PATH, 'motion_masks')  # foreground=255
CMASK_OUT_DIR = os.path.join(DATA_PATH, 'masks')          # foreground=0 (COLMAP)
os.makedirs(MASK_OUT_DIR,  exist_ok=True)
os.makedirs(CMASK_OUT_DIR, exist_ok=True)

frame_files = sorted(glob(os.path.join(IMAGES_DIR, '*.png')) +
                     glob(os.path.join(IMAGES_DIR, '*.jpg')))
print(f'Found {len(frame_files)} frames in {IMAGES_DIR}')

if len(frame_files) == 0:
    print(f'\nERROR: No images found in {IMAGES_DIR}')
    print('Check that:')
    print(f'  1. DATA_PATH is correct: {DATA_PATH}')
    print(f'  2. An images/ subdirectory exists with .png or .jpg files')
    if os.path.isdir(DATA_PATH):
        contents = os.listdir(DATA_PATH)
        print(f'  Contents of DATA_PATH: {contents}')
    else:
        print(f'  DATA_PATH does not exist!')
else:
    print(f'SAM2 repo : {SAM2_REPO_PATH}')
    print(f'SAM2 ckpt : {SAM2_CHECKPOINT}')

In [ ]:
# Load SAM2
if SAM2_REPO_PATH not in sys.path:
    sys.path.insert(0, SAM2_REPO_PATH)

from sam2.build_sam import build_sam2_video_predictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
predictor = build_sam2_video_predictor(SAM2_MODEL_CFG, SAM2_CHECKPOINT, device=device)
print(f'SAM2 loaded on {device}')

In [ ]:
# ============================================================
# Interactive point selection on the first frame
# Left click  = positive (green *)
# Right click = negative (red  x)
# Run the next cell to clear and redo.
# ============================================================

assert len(frame_files) > 0, (
    f'No frames found in {IMAGES_DIR}. '
    'Set DATA_PATH correctly in Cell 2 and re-run it first.'
)

pos_points = []   # [[x, y], ...]
neg_points = []   # [[x, y], ...]

first_frame = np.array(Image.open(frame_files[0]).convert('RGB'))

fig, ax = plt.subplots(figsize=(14, 8))
ax.imshow(first_frame)
ax.set_title('Left click: foreground (+) | Right click: background (−) | Middle click: undo last')
ax.axis('off')

scat_pos = ax.scatter([], [], c='lime',   s=120, marker='*', zorder=5, label='positive')
scat_neg = ax.scatter([], [], c='red',    s=120, marker='x', zorder=5, label='negative', linewidths=2)
ax.legend(handles=[mpatches.Patch(color='lime', label='positive (+)'),
                   mpatches.Patch(color='red',  label='negative (−)')], loc='upper right')

def _refresh_scatter():
    if pos_points:
        scat_pos.set_offsets(np.array(pos_points))
    else:
        scat_pos.set_offsets(np.empty((0, 2)))
    if neg_points:
        scat_neg.set_offsets(np.array(neg_points))
    else:
        scat_neg.set_offsets(np.empty((0, 2)))
    fig.canvas.draw_idle()

def on_click(event):
    if event.inaxes != ax or event.xdata is None:
        return
    x, y = int(round(event.xdata)), int(round(event.ydata))
    if event.button == 1:       # left  → positive
        pos_points.append([x, y])
    elif event.button == 3:     # right → negative
        neg_points.append([x, y])
    elif event.button == 2:     # middle → undo last
        if pos_points:
            pos_points.pop()
    _refresh_scatter()
    print(f'  pos={pos_points}  neg={neg_points}')

_cid = fig.canvas.mpl_connect('button_press_event', on_click)
plt.tight_layout()
plt.show()
print('Click on the image above to add points.')

In [ ]:
# Preview SAM2 prediction on frame 0 before propagating
print(f'Positive points: {pos_points}')
print(f'Negative points: {neg_points}')
assert len(pos_points) > 0, 'Add at least one positive point in the cell above.'

points_np = np.array(pos_points + neg_points, dtype=np.float32)
labels_np = np.array([1] * len(pos_points) + [0] * len(neg_points), dtype=np.int32)

# SAM2 load_video_frames only accepts JPEG files.
# If images are PNG, convert to JPEG in a temp directory.
import tempfile, shutil

_ext = os.path.splitext(frame_files[0])[1].lower()
if _ext in ('.jpg', '.jpeg'):
    SAM2_IMAGES_DIR = IMAGES_DIR
    _tmp_dir = None
else:
    print(f'Images are {_ext} — converting to JPEG for SAM2 (temp dir) ...')
    _tmp_dir = tempfile.mkdtemp(prefix='sam2_jpg_')
    for fpath in frame_files:
        stem = os.path.splitext(os.path.basename(fpath))[0]
        Image.open(fpath).convert('RGB').save(
            os.path.join(_tmp_dir, f'{stem}.jpg'), quality=95)
    SAM2_IMAGES_DIR = _tmp_dir
    print(f'  Converted {len(frame_files)} frames → {_tmp_dir}')

with torch.inference_mode(), torch.autocast(device, dtype=torch.bfloat16):
    inference_state = predictor.init_state(video_path=SAM2_IMAGES_DIR)
    predictor.reset_state(inference_state)

    frame_idx_out, obj_ids_out, mask_logits = predictor.add_new_points_or_box(
        inference_state=inference_state,
        frame_idx=0,
        obj_id=OBJ_ID,
        points=points_np,
        labels=labels_np,
    )

mask_preview = (mask_logits[0, 0] > 0).cpu().numpy()

fig2, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(first_frame)
axes[0].scatter(*zip(*pos_points), c='lime', s=100, marker='*', zorder=5)
if neg_points:
    axes[0].scatter(*zip(*neg_points), c='red', s=100, marker='x', zorder=5, linewidths=2)
axes[0].set_title('Input points')
axes[0].axis('off')

axes[1].imshow(first_frame)
axes[1].imshow(mask_preview, alpha=0.5, cmap='Greens')
axes[1].set_title('SAM2 mask preview (frame 0)')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print(f'Mask coverage: {mask_preview.mean() * 100:.1f}% of pixels')

In [ ]:
# ============================================================
# Propagate through all frames and save masks
# ============================================================
# motion_masks/  : foreground=255, background=0
# masks/         : foreground=0,   background=255  (for COLMAP --mask_path)

print('Propagating SAM2 masks through all frames ...')
saved = 0

with torch.inference_mode(), torch.autocast(device, dtype=torch.bfloat16):
    for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
        mask = (out_mask_logits[0, 0] > 0).cpu().numpy().astype(np.uint8) * 255

        frame_name = os.path.basename(frame_files[out_frame_idx])
        stem = os.path.splitext(frame_name)[0] + '.png'

        # motion_masks: foreground=255
        Image.fromarray(mask, mode='L').save(os.path.join(MASK_OUT_DIR, stem))
        # masks: foreground=0 (inverted, for COLMAP)
        Image.fromarray(255 - mask, mode='L').save(os.path.join(CMASK_OUT_DIR, stem))

        saved += 1
        if saved % 10 == 0 or saved == len(frame_files):
            print(f'  Saved {saved}/{len(frame_files)} frames ...')

print(f'Done. Saved {saved} masks to:')
print(f'  {MASK_OUT_DIR}')
print(f'  {CMASK_OUT_DIR}')

# Clean up temp JPEG directory if it was created
if _tmp_dir and os.path.isdir(_tmp_dir):
    shutil.rmtree(_tmp_dir)
    print(f'Removed temp dir: {_tmp_dir}')

In [ ]:
# ============================================================
# Verification — display a few masks overlaid on frames
# ============================================================

check_indices = [0, len(frame_files) // 4, len(frame_files) // 2,
                 3 * len(frame_files) // 4, len(frame_files) - 1]

fig3, axes = plt.subplots(1, len(check_indices), figsize=(4 * len(check_indices), 4))
for ax_i, idx in enumerate(check_indices):
    frame = np.array(Image.open(frame_files[idx]).convert('RGB'))
    stem  = os.path.splitext(os.path.basename(frame_files[idx]))[0] + '.png'
    mpath = os.path.join(MASK_OUT_DIR, stem)
    if os.path.exists(mpath):
        msk = np.array(Image.open(mpath).convert('L'))
        axes[ax_i].imshow(frame)
        axes[ax_i].imshow(msk, alpha=0.4, cmap='Greens', vmin=0, vmax=255)
        axes[ax_i].set_title(f'frame {idx:05d}')
    else:
        axes[ax_i].set_title(f'frame {idx:05d} (missing)')
    axes[ax_i].axis('off')

plt.suptitle('Motion mask overlay (green = foreground)', fontsize=13)
plt.tight_layout()
plt.show()